# SwitcherLLM

Mini programa educativo en Python que envía el **mismo prompt** a OpenAI, Anthropic (Claude), Google Gemini y DeepSeek, y muestra las diferencias entre sus APIs.

Este notebook es una versión **autocontenida y paso a paso** del proyecto: cada celda de código es el contenido real de un archivo fuente (no una copia), y las celdas de texto explican qué hace y por qué.

## Objetivo educativo

No busca ser una herramienta productiva, sino mostrar de forma mínima:

- La **estructura básica** de una llamada a la API de cada proveedor.
- Que cada API tiene **forma distinta** (Anthropic separa el system prompt, Gemini usa objetos `Content`, DeepSeek es compatible con OpenAI...).
- Cómo se manejan **credenciales** (`.env`, variables de entorno) de forma segura.
- Cómo se **normalizan errores** y se reintenta con backoff exponencial.
- Cómo calcular el **coste estimado** de cada llamada (LLMPrice, normalización de tokens por proveedor, tabla local de precios).
- Cómo integrar **monitoreo de errores** en la nube (Sentry, init con DSN, captura de excepciones).

## Patrones y técnicas usadas

| Patrón / técnica | Dónde | Qué resuelve |
| --- | --- | --- |
| **Adapter** | `providers/*_provider.py` | Cada API es distinta; un adaptador traduce a una interfaz común |
| **Factory** | `providers/factory.py` | Crear el adaptador correcto solo con el nombre del proveedor |
| **Strategy** | `switcherllm.py` (`LLMClient`) | Cambiar de proveedor en tiempo de ejecución |
| **Error normalizado** | `providers/errors.py` | Un solo tipo de excepción para los 4 proveedores |
| **Backoff exponencial** | `providers/errors.py` | Reintentar solo errores temporales (rate limit) |
| **Normalización de usage** | adaptadores | Unificar `input_tokens`/`output_tokens` |
| **Coste estimado** | `providers/pricing.py` | Estimar USD por llamada con tabla local de LLMPrice |
| **Monitoreo** | `switcherllm.py` (Sentry) | Reportar errores en la nube si hay `SENTRY_DSN` |

## 0. Entorno y credenciales

Antes de ejecutar nada: crear un archivo `.env` en la **carpeta del proyecto** (donde vive este notebook) copiando `.env.example`, y rellenar al menos una API key:

```bash
cp .env.example .env   # y editar las claves
```

| Variable | Ingresar |
| --- | --- |
| `OPENAI_API_KEY` | https://platform.openai.com/api-keys |
| `ANTHROPIC_API_KEY` | https://console.anthropic.com/settings/keys |
| `GEMINI_API_KEY` | https://aistudio.google.com/app/apikey |
| `DEEPSEEK_API_KEY` | https://platform.deepseek.com/api_keys |
| `LLM_DEFAULT` | Proveedor inicial (por defecto `anthropic`) |
| `LLM_ROLE` | Rol/system prompt inicial del asistente (opcional) |
| `SENTRY_DSN` | URL del proyecto Sentry (opcional; vacío = desactivado) |

Los secretos **nunca** van en el código: se leen del entorno con `os.environ.get(...)`.

In [ ]:
# Cargamos las claves del archivo .env de la carpeta del proyecto.
# En el paquete real esto lo hace providers/__init__.py al importar;
# aquí lo hacemos explícito porque el notebook debe ser autocontenido.
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path, override=False)
    print(f"  Entorno cargado desde {env_path}")
else:
    print("  No encontré .env en el directorio de trabajo; las llamadas fallarán.")

## 1. La interfaz común: `providers/base.py`

Es el *target* del patrón Adapter: un contrato que **todo** adaptador debe cumplir, para que el código cliente nunca dependa de un SDK concreto.

- `Role` y `Message`: un turno de conversación (system / user / assistant).
- `LLMResponse`: respuesta **normalizada** (`content`, `model`, `usage`).
- `BaseProvider(ABC)`: métodos abstractos `get_available_models()` y `chat()`.

In [ ]:
"""Interfaz común (el "target" del patrón Adapter).

Aquí se define la forma en la que TODO el resto del programa
habla con cualquier LLM, sin importar el proveedor real.
"""

from abc import ABC, abstractmethod  # Para crear clases "esqueleto" que obligan a implementar métodos
from dataclasses import dataclass  # Genera __init__, __repr__, etc. automáticamente
from typing import Literal

# Un rol solo puede ser uno de estos tres valores (ayuda de tipos)
Role = Literal["system", "user", "assistant"]


@dataclass
class Message:
    """Un turno de conversación, igual que en los chats."""
    role: Role      # quién habla (sistema, usuario o asistente)
    content: str    # el texto del mensaje


@dataclass
class LLMResponse:
    """Respuesta normalizada que devuelven todos los adaptadores."""
    content: str                 # el texto que respondió el LLM
    model: str                   # qué modelo respondió
    usage: dict | None = None    # consumo de tokens (opcional)


class BaseProvider(ABC):
    """Contrato que toda implementación (adaptador) debe cumplir."""

    name: str = "base"  # nombre corto con el que se registra en el factory

    @abstractmethod
    def get_available_models(self) -> list[str]:
        """Devuelve la lista de modelos disponibles (consulta en vivo)."""
        pass

    @abstractmethod
    def chat(self, messages: list[Message], model: str, **kwargs) -> LLMResponse:
        """Envía una conversación y devuelve la respuesta del LLM."""
        pass

## 2. Errores normalizados y reintentos: `providers/errors.py`

Cada SDK lanza sus propias excepciones. En lugar de propagarlas, cada adaptador las **envuelve** en un `LLMError` con el nombre del proveedor y el código HTTP.

- `RateLimitError` (temporal) → **sí** se reintenta.
- `AuthenticationError` / resto (permanente) → **no** se reintenta.

El decorador `@retry_with_backoff(...)` espera 1s, 2s, 4s... antes de cada reintento (backoff exponencial) hasta `max_retries` veces.

In [ ]:
"""Manejo de errores y reintentos con backoff exponencial."""

import functools  # Conserva el nombre/metadatos de la función al decorarla
import time
from typing import Callable


class LLMError(Exception):
    """Error base: cualquier fallo de un proveedor se envuelve en este tipo.

    Normalizar los errores permite tratarlos igual en los 4 adaptadores.
    """

    def __init__(self, provider: str, message: str, status_code: int | None = None):
        self.provider = provider      # qué proveedor falló (openai, gemini...)
        self.message = message        # descripción del fallo
        self.status_code = status_code  # código HTTP si la API lo devolvió
        # Mensaje legible: "[openai] API key inválida (HTTP 401)"
        super().__init__(f"[{provider}] {message} (HTTP {status_code})" if status_code else f"[{provider}] {message}")


class RetryableError(LLMError):
    """Marca los errores que SÍ merecen reintentarse (problemas temporales)."""
    pass


class RateLimitError(RetryableError):
    """El proveedor nos pide esperar: nos limitaron por cuota de peticiones."""
    pass


class AuthenticationError(LLMError):
    """Key inválida/prohibida: retintentar no sirve, hay que arreglar la credencial."""
    pass


def retry_with_backoff(
    max_retries: int = 3,
    base_delay: float = 1.0,
    backoff_factor: float = 2.0,
) -> Callable:
    """Decorator: reintenta una función que llama a un LLM.

    - max_retries: cuántas veces reintentar antes de rendirse.
    - base_delay: segundos a esperar antes del primer reintento.
    - backoff_factor: multiplicador exponencial (1s, 2s, 4s...).

    Un decorador "envuelve" una función: se usa como @retry_with_backoff(...)
    sobre cualquier función que trate con la red.
    """

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = base_delay
            attempt = 0
            while True:
                try:
                    return func(*args, **kwargs)  # la llamada original funcionó
                except RateLimitError as e:
                    # Error temporal: esperamos y volvemos a intentar
                    attempt += 1
                    if attempt > max_retries:
                        raise LLMError(e.provider, f"Rate limit persistente tras {max_retries} reintentos") from e
                    print(f"  -> Rate limit, reintento {attempt}/{max_retries} en {delay:.1f}s")
                    time.sleep(delay)
                    delay *= backoff_factor  # cada intento espera más (backoff)
                except (AuthenticationError, LLMError) as e:
                    # Errores no recuperables: no tiene sentido reintentar
                    raise e

        return wrapper

    return decorator

## 3. Coste estimado: `providers/pricing.py`

Las APIs no devuelven el coste; LLMPrice trae un **snapshot local** de los precios por millón de tokens (funciona offline). La versión del paquete codifica la fecha de ese snapshot (`2026.4.3` = 3 de abril de 2026).

`estimate_cost(model, usage)` devuelve `(coste_en_USD, nota)`. La nota explica la fuente o, si no se pudo estimar (modelo ausente del snapshot o sin metadatos de tokens), **por qué**.

In [ ]:
"""Estimación del coste de una llamada LLM usando la base de precios LLMPrice.

¿De dónde salen los precios?
- Las APIs no los devuelven; cada proveedor publica precio por millón de tokens (#/1M)
  de entrada y salida.
- LLMPrice trae un snapshot local de esos precios (funciona offline). La versión
  del paquete codifica la fecha de ese snapshot (p. ej. 2026.4.3 = 3 de abril de 2026).
"""

import importlib.metadata  # para leer la versión instalada del paquete

from llmprice import LLMPrice

# Se crea UNA vez y se reutiliza (el snapshot se carga en memoria)
_price_db = LLMPrice()
_snapshot_version = importlib.metadata.version("llmprice-kit")

_MESES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre",
]


def _snapshot_label() -> str:
    """Fecha legible del snapshot, p. ej. '2026.4.3 (de abril)'."""
    try:
        ano, mes, dia = _snapshot_version.split(".")  # formato AAAA.M.D
        return f"{_snapshot_version} (de {_MESES[int(mes) - 1]})"
    except (ValueError, IndexError):
        return _snapshot_version  # si el formato no es el esperado, solo la versión


def estimate_cost(model: str, usage: dict | None) -> tuple[float | None, str]:
    """Devuelve (coste_estimado_en_USD, nota).

    La nota explica la fuente o, si coste es None, POR QUÉ no se pudo estimar
    (modelo ausente del snapshot, o falta de metadatos de tokens).

    Fórmula: (input_tokens * precio_input + output_tokens * precio_output) / 1M.
    """
    if not usage:
        return None, "no hay metadatos de tokens en la respuesta"
    try:
        info = _price_db.get(model)  # excepción si el modelo no está en la base
    except KeyError:
        return None, f"LLMPrice no tiene datos para '{model}' en el snapshot {_snapshot_label()}"

    input_tokens = usage.get("input_tokens") or 0
    output_tokens = usage.get("output_tokens") or 0
    cost = (
        input_tokens * info.input_cost_per_1m
        + output_tokens * info.output_cost_per_1m
    ) / 1_000_000
    return cost, f"LLMPrice (snapshot {_snapshot_label()})"

## 4. Los adaptadores

Cuatro clases que cumplen el contrato de `BaseProvider` pero cada una usando su SDK. Fíjate en cómo cambia la estructura de la llamada en cada API y en cómo los errores del SDK se mapean a `LLMError`.

### 4.1 OpenAI (`providers/openai_provider.py`)

- Llamada: `client.chat.completions.create(...)` con `messages` de `role` + `content`.
- Tokens: `usage.prompt_tokens` / `usage.completion_tokens` → normalizados a `input`/`output`.
- Gestiona `/models` filtrando solo modelos de chat/texto.

In [ ]:
"""Patrón Adapter: traduce la API de OpenAI a la interfaz común BaseProvider."""

import os  # Para leer la API key de las variables de entorno

# Importamos el SDK oficial de OpenAI y sus errores propios
from openai import OpenAI, AuthenticationError, RateLimitError, APIStatusError

# Nota del notebook: Message, LLMResponse, BaseProvider y los tipos de error
# normalizados ya quedaron definidos en las celdas superiores (base.py y errors.py).


class OpenAIProvider(BaseProvider):
    """Adaptador para OpenAI: cumple el contrato de BaseProvider usando el SDK de OpenAI."""

    name = "openai"  # con este nombre se registra en el factory

    def __init__(self, api_key: str | None = None, model: str = "gpt-4.1-mini"):
        # Si no nos pasan key, la buscamos en el entorno (.env). Es un patrón de seguridad:
        # los secretos NUNCA van en el código fuente.
        if not api_key:
            api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            # "fail fast": mejor detenerse aquí que fallar raro a mitad de una llamada
            raise LLMError(self.name, "Falta OPENAI_API_KEY")
        self.client = OpenAI(api_key=api_key)  # cliente autenticado del SDK
        self.model = model  # modelo por defecto de este adaptador

    def get_available_models(self) -> list[str]:
        """Consulta en vivo el endpoint /models de OpenAI."""
        try:
            ids = [m.id for m in self.client.models.list()]  # petición HTTP real al servidor
        except APIStatusError as e:
            # Cualquier fallo de la API se envuelve en nuestro error normalizado
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e
        # La API devuelve todo: embeddings, TTS, imágenes, etc.
        # Se filtran solo los modelos de chat/texto razonado.
        excluded = ("image", "audio", "realtime", "transcribe", "tts",
                    "whisper", "sora", "embedding", "moderation", "search",
                    "codex", "davinci", "babbage", "instruct")
        return sorted(
            m for m in ids
            if not any(flag in m for flag in excluded)  # se queda si no es excluido
        )

    def chat(self, messages: list[Message], model: str | None = None, temperature: float = 0.7) -> LLMResponse:
        """Estructura típica de una llamada a OpenAI usando la RESPONSES API (SDK).

        La Responses API es la recomendada por OpenAI para proyectos nuevos.
        La antigua Chat Completions API queda comentada más abajo (no se elimina).
        """
        model = model or self.model  # si no pasan modelo, usamos el por defecto
        try:
            # ─── VERSIÓN ANTIGUA (deprecada): Chat Completions API ───
            # Soportada indefinidamente, pero ya no se recomienda para nuevos desarrollos.
            #   response = self.client.chat.completions.create(
            #       model=model,
            #       messages=[{"role": m.role, "content": m.content} for m in messages],
            #       temperature=temperature,  # 0 = determinista, 1 = creativo
            #   )
            #   return LLMResponse(
            #       content=response.choices[0].message.content or "",
            #       model=response.model,
            #       usage={  # Chat Completions usa prompt/completion_tokens
            #           "input_tokens": response.usage.prompt_tokens if response.usage else None,
            #           "output_tokens": response.usage.completion_tokens if response.usage else None,
            #       },
            #   )

            # ─── VERSIÓN NUEVA: Responses API (recomendada por OpenAI) ───
            # La petición clave: enviamos los mensajes con roles "system"/"user"/"assistant"
            response = self.client.responses.create(
                model=model,
                input=[{"role": m.role, "content": m.content} for m in messages],
                temperature=temperature,  # 0 = determinista, 1 = creativo
            )
            # Normalizamos la respuesta del SDK a nuestro formato común (LLMResponse)
            return LLMResponse(
                content=response.output_text or "",
                model=response.model,
                usage={  # la Responses API ya usa input_tokens/output_tokens directos
                    "input_tokens": response.usage.input_tokens if response.usage else None,
                    "output_tokens": response.usage.output_tokens if response.usage else None,
                },
            )
        # Mapeamos los errores específicos del SDK a nuestros errores normalizados
        # para que quien llama al adaptador no dependa del SDK concreto.
        except AuthenticationError as e:
            raise AppAuthError(self.name, "API key inválida o sin permisos", getattr(e, "status_code", 401)) from e
        except RateLimitError as e:
            raise AppRateLimitError(self.name, "Rate limit superado", getattr(e, "status_code", 429)) from e
        except APIStatusError as e:
            # Cualquier otro fallo HTTP (404 modelo inexistente, 500 del servidor...)
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e

### 4.2 Anthropic / Claude (`providers/anthropic_provider.py`)

- Llamada DISTINTA: el **system prompt viaja aparte**, en el parámetro `system` (obligatorio pasar lista, `[]` si no hay).
- Exige fijar `max_tokens`.
- Tokens: `usage.input_tokens` / `output_tokens` (ya normalizados).

In [ ]:
"""Patrón Adapter: traduce la API de Anthropic (Claude) a la interfaz común BaseProvider."""

import os  # Para leer la API key de las variables de entorno

import anthropic  # SDK oficial de Anthropic
# Importamos los errores propios del SDK
from anthropic import AuthenticationError, RateLimitError, APIStatusError

# Nota del notebook: Message, LLMResponse, BaseProvider y los tipos de error
# normalizados ya quedaron definidos en las celdas superiores (base.py y errors.py).


class AnthropicProvider(BaseProvider):
    """Adaptador para Anthropic (Claude): cumple el contrato de BaseProvider usando el SDK de Anthropic."""

    name = "anthropic"  # con este nombre se registra en el factory

    def __init__(self, api_key: str | None = None, model: str = "claude-haiku-4-5-20251001"):
        # Mismo patrón de seguridad que OpenAI: key del entorno, nunca en el código
        if not api_key:
            api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise LLMError(self.name, "Falta ANTHROPIC_API_KEY")
        self.client = anthropic.Anthropic(api_key=api_key)  # cliente autenticado del SDK
        self.model = model  # modelo por defecto de este adaptador

    def get_available_models(self) -> list[str]:
        """Consulta en vivo el endpoint /models de Anthropic."""
        try:
            # .data: la lista de modelos viene dentro de una estructura paginada
            return [m.id for m in self.client.models.list(limit=50).data]
        except anthropic.APIStatusError as e:
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e

    def chat(self, messages: list[Message], model: str | None = None, max_tokens: int = 1024) -> LLMResponse:
        """Estructura típica de una llamada al API de Anthropic (SDK).

        Dato de aprendizaje: la API de Anthropic es DISTINTA a la de OpenAI.
        La conversación se envía en messages, pero el "system prompt" viaja
        aparte, en el parámetro system. Por eso este adaptador lo separa.
        """
        model = model or self.model
        try:
            # Anthropic separa el system prompt del resto de mensajes
            system_text = "\n".join(m.content for m in messages if m.role == "system")
            user_msgs = [  # aquí solo van user/assistant, como dicts
                {"role": m.role, "content": m.content}
                for m in messages
                if m.role in ("user", "assistant")
            ]
            response = self.client.messages.create(
                model=model,
                max_tokens=max_tokens,  # Anthropic exige fijar el tope de tokens a generar
                system=[{"type": "text", "text": system_text}] if system_text else [],
                messages=user_msgs,
            )
            # La respuesta trae bloques; juntamos solo los de texto
            text = "".join(
                block.text for block in response.content if block.type == "text"
            )
            # Normalizamos la respuesta del SDK a nuestro formato común (LLMResponse)
            return LLMResponse(
                content=text,
                model=response.model,
                usage={  # cuidado: Anthropic lo llama input/output, no prompt/completion
                    "input_tokens": response.usage.input_tokens,
                    "output_tokens": response.usage.output_tokens,
                },
            )
        # Mapeamos los errores del SDK a nuestros errores normalizados
        except AuthenticationError as e:
            raise AppAuthError(self.name, "API key inválida o sin permisos", getattr(e, "status_code", 401)) from e
        except RateLimitError as e:
            raise AppRateLimitError(self.name, "Rate limit superado", getattr(e, "status_code", 429)) from e
        except APIStatusError as e:
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e

### 4.3 Google Gemini (`providers/gemini_provider.py`)

- La conversación se modela con objetos `types.Content` (rol + partes) y los parámetros van en un `config`.
- El system prompt viaja como `system_instruction` en el config, fuera de los contents.
- Tokens: `usage_metadata.prompt_token_count` / `candidates_token_count`.
- Errores especiales: `PERMISSION_DENIED` → auth, `RESOURCE_EXHAUSTED` → rate limit.

In [ ]:
"""Patrón Adapter: traduce la API de Google Gemini a la interfaz común BaseProvider."""

import logging
import os

from google import genai  # SDK oficial de Google Gemini


# El SDK emite un logger.warning sobre AFC (automatic function calling).
# Solo aplica cuando se pasan herramientas; para texto simple es ruido,
# así que subimos el nivel de logging para silenciarlo.
logging.getLogger("google_genai.models").setLevel(logging.ERROR)
from google.genai import types  # tipos del SDK (Content, Part, configs de generación)
from google.genai.errors import APIError, ServerError  # errores propios del SDK


class GeminiProvider(BaseProvider):
    """Adaptador para Google Gemini: cumple el contrato de BaseProvider usando el SDK de Gemini."""

    name = "gemini"  # con este nombre se registra en el factory

    def __init__(self, api_key: str | None = None, model: str = "gemini-3.6-flash"):
        # Mismo patrón de seguridad que los demás adaptadores: key del entorno
        if not api_key:
            api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise LLMError(self.name, "Falta GEMINI_API_KEY")
        self.client = genai.Client(api_key=api_key)  # cliente autenticado del SDK
        self.model = model  # modelo por defecto de este adaptador

    def get_available_models(self) -> list[str]:
        """Consulta en vivo el endpoint /models de Gemini.

        La API devuelve TODOS los modelos (imagen, audio, vídeo, TTS,
        robótica...). Se filtra a texto/chat: solo los 'gemini-*' y 'gemma-*'
        sin sufijos de tareas no textuales.
        """
        try:
            # Quitamos el prefijo "models/" que la API añade a cada nombre
            names = [m.name.removeprefix("models/") for m in self.client.models.list()]
        except APIError as e:
            raise LLMError(self.name, str(e), getattr(e, "code", None)) from e
        # Lista de subcadenas que delatan modelos NO textuales
        excluded = ("image", "audio", "live", "tts", "transcribe", "embedding",
                    "veo", "lyria", "robotics", "aqa", "nano-banana",
                    "computer-use", "deep-research", "antigravity")
        return sorted(
            n for n in names
            if (n.startswith("gemini-") or n.startswith("gemma-"))  # solo chat
            and not any(flag in n for flag in excluded)
        )

    def chat(self, messages: list[Message], model: str | None = None, temperature: float = 0.7) -> LLMResponse:
        """Estructura típica de una llamada a Google Gemini (SDK).

        Dato de aprendizaje: Gemini modela la conversación con objetos
        types.Content (rol + partes). El system prompt NO va en la lista
        de mensajes: viaja aparte, como system_instruction en el config.
        """
        model = model or self.model
        # Gemini usa el modelo de "transacción" para system prompts
        # Convierte nuestros Message a types.Content (roles "user"/"model")
        contents = [
            types.Content(role="model" if m.role == "assistant" else "user", parts=[types.Part(text=m.content)])
            for m in messages
            if m.role != "system"
        ]
        # El system prompt se separa y se pasa como instrucción de sistema
        system_parts = [types.Part(text=m.content) for m in messages if m.role == "system"]
        try:
            response = self.client.models.generate_content(
                model=model,
                contents=contents,
                config=types.GenerateContentConfig(  # los parámetros van en un "config"
                    system_instruction=system_parts if system_parts else None,
                    temperature=temperature,
                ),
            )
            # Normalizamos la respuesta del SDK a nuestro formato común (LLMResponse)
            return LLMResponse(
                content=response.text or "",
                model=response.model_version,  # metadato propio de la respuesta
                usage={  # el SDK expone el gasto en usage_metadata; normalizamos nombres
                    "input_tokens": response.usage_metadata.prompt_token_count if response.usage_metadata else None,
                    "output_tokens": response.usage_metadata.candidates_token_count if response.usage_metadata else None,
                },
            )
        except ServerError as e:
            # Gemini incluye el motivo en el texto del error ("PERMISSION_DENIED"...)
            code = e.code if hasattr(e, "code") else None
            message = str(e)
            if "PERMISSION_DENIED" in message or (code and code == 403):
                raise AppAuthError(self.name, "API key inválida o sin permisos", code or 403) from e
            if "RESOURCE_EXHAUSTED" in message or (code and code == 429):
                raise AppRateLimitError(self.name, "Rate limit superado (RESOURCE_EXHAUSTED)", code or 429) from e
            raise LLMError(self.name, message, code) from e
        except APIError as e:
            raise LLMError(self.name, str(e), getattr(e, "code", None)) from e


### 4.4 DeepSeek (`providers/deepseek_provider.py`)

- Dato CLAVE: DeepSeek usa una **API compatible con OpenAI**. El adaptador reutiliza el SDK de OpenAI y solo cambia `base_url` a `https://api.deepseek.com`.

In [ ]:
"""Patrón Adapter: traduce la API de DeepSeek a la interfaz común BaseProvider.

Dato de aprendizaje CLAVE: DeepSeek usa una API compatible con OpenAI.
El adaptador NO necesita un SDK propio: reutiliza el SDK de OpenAI
y solo cambia la URL base a la que apunta.
"""

import os

from openai import OpenAI, AuthenticationError, RateLimitError, APIStatusError



class DeepSeekProvider(BaseProvider):
    """Adaptador para DeepSeek. Solo cambia base_url del cliente de OpenAI."""

    name = "deepseek"  # con este nombre se registra en el factory

    BASE_URL = "https://api.deepseek.com"  # la única diferencia real con OpenAI

    def __init__(self, api_key: str | None = None, model: str = "deepseek-v4-flash"):
        # Mismo patrón de seguridad que los demás: key del entorno, nunca en código
        if not api_key:
            api_key = os.environ.get("DEEPSEEK_API_KEY")
        if not api_key:
            raise LLMError(self.name, "Falta DEEPSEEK_API_KEY")
        # Cliente de OpenAI apuntando al servidor de DeepSeek
        self.client = OpenAI(api_key=api_key, base_url=self.BASE_URL)
        self.model = model

    def get_available_models(self) -> list[str]:
        """Consulta en vivo /models vía el endpoint compatible con OpenAI."""
        try:
            return sorted(m.id for m in self.client.models.list())
        except APIStatusError as e:
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e

    def chat(self, messages: list[Message], model: str | None = None, temperature: float = 0.7) -> LLMResponse:
        """Estructura de llamada: IDÉNTICA a OpenAI (sirve este mismo SDK)."""
        model = model or self.model
        try:
            # Misma forma que en openai_provider: chat.completions.create
            response = self.client.chat.completions.create(
                model=model,
                messages=[{"role": m.role, "content": m.content} for m in messages],
                temperature=temperature,
            )
            # Normalizamos la respuesta del SDK a nuestro formato común (LLMResponse)
            return LLMResponse(
                content=response.choices[0].message.content or "",
                model=response.model,
                usage={  # mismo esquema de tokens que OpenAI; normalizamos a input/output
                    "input_tokens": response.usage.prompt_tokens if response.usage else None,
                    "output_tokens": response.usage.completion_tokens if response.usage else None,
                },
            )
        # Mapeamos los errores del SDK a nuestros errores normalizados
        except AuthenticationError as e:
            raise AppAuthError(self.name, "API key inválida o sin permisos", getattr(e, "status_code", 401)) from e
        except RateLimitError as e:
            raise AppRateLimitError(self.name, "Rate limit superado", getattr(e, "status_code", 429)) from e
        except APIStatusError as e:
            raise LLMError(self.name, str(e), getattr(e, "status_code", None)) from e

## 5. El factory: `providers/factory.py`

Un único punto que crea el adaptador correcto por nombre. Sin él, el código cliente tendría un `if/elif` por proveedor y conocería cada SDK.

- `register(nombre, clase)`: alta de proveedores (configuración, aquí los 4).
- `create(nombre)` → instancia del adaptador; tolera mayúsculas; error claro si el nombre no existe.
- Añadir un proveedor nuevo NO implica tocar este diccionario.

In [ ]:
"""Patrón Factory: un único punto que crea el adaptador correcto.

Sin esto, el código cliente tendría que hacer un if/elif por proveedor
y conocer cada SDK. Con el factory solo pide un nombre.
"""

# Nota del notebook: BaseProvider y las 4 clases concretas (OpenAIProvider,
# AnthropicProvider, GeminiProvider, DeepSeekProvider) ya quedaron definidas
# en las celdas superiores, así que no hace falta importarlas.


class ProviderFactory:
    """Patrón Factory: crea el adaptador correcto según el nombre.

    Así el código cliente no necesita saber qué SDK usar:
    solo pide un proveedor por nombre y recibe la instancia.
    """
    # Registro de proveedores: nombre -> clase del adaptador.
    # Crecer con más proveedores NO implica tocar este diccionario.
    _registry: dict[str, type[BaseProvider]] = {}

    @classmethod
    def register(cls, name: str, provider_class: type[BaseProvider]) -> None:
        """Añade un proveedor nuevo al catálogo del factory (diccionario)."""
        cls._registry[name] = provider_class

    @classmethod
    def create(cls, name: str, **kwargs) -> BaseProvider:
        """Devuelve una instancia ya construida del adaptador pedido.

        - name: "openai", "anthropic", "gemini" o "deepseek"
        - **kwargs: opcional, p.ej. pasar la api_key manualmente.
        """
        name = name.lower()  # toleramos mayúsculas: "OpenAI" -> "openai"
        if name not in cls._registry:
            # Fallo claro y con pista, mejor que un KeyError críptico
            available = ", ".join(sorted(cls._registry))
            raise ValueError(f"Proveedor desconocido '{name}'. Disponibles: {available}")
        return cls._registry[name](**kwargs)  # llama a la clase -> construye el adaptador

    @classmethod
    def list_available(cls) -> list[str]:
        """Devuelve los nombres de proveedores registrados, ordenados."""
        return sorted(cls._registry)


# Alta de los 4 proveedores al arrancar el módulo (es la "configuración")
ProviderFactory.register(OpenAIProvider.name, OpenAIProvider)
ProviderFactory.register(AnthropicProvider.name, AnthropicProvider)
ProviderFactory.register(GeminiProvider.name, GeminiProvider)
ProviderFactory.register(DeepSeekProvider.name, DeepSeekProvider)

## 6. El código cliente y el REPL: `switcherllm.py`

- `LLMClient` (Strategy): guarda el proveedor actual, lo cambia con `switch()` en caliente, y define la conversación con `ask()`. Está decorado con `@retry_with_backoff`.
- `ask()` imprime la respuesta, el **modelo real** que respondió, los **tokens** y el **coste estimado** (o el motivo por el que no se pudo estimar).
- `main()` / `run()`: consola interactiva (REPL) con los comandos `modelos`, `switch <proveedor>`, `role <texto>` y `salir`. Activa Sentry solo si existe `SENTRY_DSN`.

> En este notebook no arrancamos el REPL (necesita `input()`); el guard `if __name__ == "__main__"` se ha sustituido por un comentario. Para la consola usa `switcherllm` o `python3 switcherllm.py`.

In [ ]:
"""SwitcherLLM: llama al mismo prompt con diferentes LLMs.

Patrones usados:
- Factory: crea el adaptador por nombre de proveedor.
- Strategy: el "current" objeto cambia el proveedor en tiempo de ejecución.
- Adapter: cada proveedor expone una interfaz común (chat()).

Este archivo es el CÓDIGO CLIENTE (y punto de entrada del programa):
solo conoce abstracciones (BaseProvider, ProviderFactory),
nunca los SDK concretos.

Punto de entrada:  python3 switcherllm.py
"""

import os

import sentry_sdk  # monitoreo de errores en la nube (Sentry.io)

# Nota del notebook: ProviderFactory, Message, LLMError, retry_with_backoff
# y estimate_cost ya quedaron definidos en las celdas superiores.


class LLMClient:
    """Patrón Strategy: el proveedor actual es intercambiable en runtime.

    El "contexto" (esta clase) guarda un objeto provider. Ese objeto,
    la estrategia, se puede sustituir en caliente sin tocar este código.
    """

    def __init__(self, provider_name: str, system_prompt: str = None, **provider_kwargs):
        # La fábrica construye el adaptador; aquí solo almacenamos una referencia
        self.current = ProviderFactory.create(provider_name, **provider_kwargs)
        # Rol del asistente (system prompt): default o el pasado por el llamador
        self.system_prompt = system_prompt or "Eres un asistente breve que responde en español."

    def set_role(self, system_prompt: str) -> None:
        # Define el rol (system prompt) que verá el LLM antes del mensaje del usuario
        self.system_prompt = system_prompt

    def switch(self, provider_name: str, **provider_kwargs) -> None:
        # Cambia la estrategia en caliente: se reemplaza el objeto provider
        self.current = ProviderFactory.create(provider_name, **provider_kwargs)

    def list_models(self) -> list[str]:
        # El cliente nunca sabe qué SDK hay detrás: solo conoce esta interface
        return self.current.get_available_models()

    @retry_with_backoff(max_retries=3, base_delay=1.0)  # reintentos ante rate limit
    def ask(self, prompt: str, **kwargs) -> str:
        message = [
            Message(role="system", content=self.system_prompt),
            Message(role="user", content=prompt),
        ]
        resp = self.current.chat(message, **kwargs)  # misma conversación, cualquier proveedor
        print(f"  {resp.content}\n")

        if resp.usage:
            print(f"  Tokens -> {resp.usage}")
        # El modelo que respondió lo trae la propia respuesta normalizada
        print(f"  Modelo -> {resp.model}")
        # estimate_cost devuelve (coste, nota): la nota explica la fuente o el motivo del fallo
        cost, cost_note = estimate_cost(resp.model, resp.usage)
        if cost is not None:
            print(f"  Coste estimado -> ${cost:.6f}\n")
        else:
            print(f"  Coste NO estimado -> {cost_note}\n")
        return resp.content


def main(client: LLMClient) -> None:
    print("Proveedores registrados en el factory:")
    for name in ProviderFactory.list_available():
        # Instanciar da acceso al modelo default sin llamar a la API
        try:
            provider = ProviderFactory.create(name)
            print(f"  - {name} [{provider.model}]")
        except LLMError as e:
            print(f"  - {name} [sin key]")  # sin API key configurada para ese proveedor
    print()
    # Rol por defecto definido al iniciar (variable LLM_ROLE o su fallback)
    print(f"Rol por defecto: {client.system_prompt}")
    print()

    available = ProviderFactory.list_available()

    # Bucle de la consola interactiva (REPL simple)
    while True:
        print(f"Proveedor actual: {client.current.name}")
        print("Proveedores disponibles: " + ", ".join(available))
        prompt = input("\nPrompt (o 'modelos', 'switch <proveedor>', 'role <texto>', 'salir'): ").strip()

        if prompt.lower() == "salir":
            break
        if prompt.lower() == "exit":
            break
        if prompt.lower() in ("modelos", "models", "list"):
            # Comando interno: NO hace falta llamar al LLM, es endpoint /models
            print(f"  Modelos de {client.current.name}: {client.list_models()}\n")
            continue
        if prompt.startswith("switch "):
            # Cambia la estrategia en tiempo de ejecución (patrón Strategy)
            target = prompt.split()[1].lower()
            if target in available:
                client.switch(target)
                print(f"-> Cambiado a {target}\n")
            else:
                print(f"Proveedor '{target}' no existe.\n")
            continue
        if prompt.lower() in ("role", "rol", "system"):
            # Muestra el rol actual sin llamar al LLM
            print(f"  Rol actual: {client.system_prompt}\n")
            continue
        if prompt.lower().startswith(("role ", "rol ")):
            # Cambia el rol (system prompt) en caliente: afecta a la próxima consulta
            client.set_role(prompt.split(" ", 1)[1].strip())
            print(f"-> Rol actualizado: {client.system_prompt}\n")
            continue
        if not prompt:
            continue  # entrada vacía: vuelve a esperar

        # Entrada libre: la tratamos como un prompt para el LLM activo
        print(f"\n>>> Consultando a: {client.current.name}")
        try:
            client.ask(prompt)
        except LLMError as e:
            # Enviamos el error a Sentry (los LLMError también se reportan)
            sentry_sdk.capture_exception(e)
            print(f"  ERROR: {e}\n")
        except Exception as e:
            # Errores inesperados: a Sentry y se siguen mostrando en local
            sentry_sdk.capture_exception(e)
            print(f"  ERROR inesperado: {e}\n")


def run() -> None:
    """Punto de entrada real del programa.

    Se invoca tanto desde la consola (comando 'switcherllm' tras pip install)
    como al ejecutar 'python3 switcherllm.py'.
    """
    # Sentry: solo se activa si existe SENTRY_DSN en el entorno (.env)
    if dsn := os.environ.get("SENTRY_DSN"):
        sentry_sdk.init(dsn=dsn, environment="development", traces_sample_rate=1.0)
        print("  -> Sentry habilitado: los errores se reportan a Sentry.io\n")

    # Se lee la variable de entorno LLM_DEFAULT si existe, si no anthropic
    default_provider = os.environ.get("LLM_DEFAULT", "anthropic")
    # Se lee la variable de entorno LLM_ROLE si existe, si no el rol por defecto
    default_role = os.environ.get("LLM_ROLE", "Eres un asistente breve que responde en español.")
    # el cliente se crea UNA vez y se reutiliza
    client = LLMClient(default_provider, system_prompt=default_role)
    print(f"\nIniciando SwitcherLLM con proveedor default: {default_provider}\n")
    main(client)


if __name__ == "__main__":
    # Este guard: el código solo corre si ejecutamos este script directamente,
    # no si se importa desde otro archivo.
    run()

## 7. Demo guiada

Recreamos el arranque de `run()` (Sentry + proveedor/rol por defecto) y hacemos tres cosas:

1. Listar modelos del proveedor activo (consulta en vivo).
2. Una pregunta real al proveedor por defecto.
3. Cambiar de proveedor en caliente (`switch()`, patrón Strategy) y repetir.

Si una key falta, verás un `LLMError` cómodo en vez de un crash.

In [ ]:
# Recomponemos el arranque de run() y hacemos una demo guiada.
import os

if dsn := os.environ.get("SENTRY_DSN"):
    sentry_sdk.init(dsn=dsn, environment="development", traces_sample_rate=1.0)
    print("  -> Sentry habilitado: los errores se reportan a Sentry.io\n")

print("Proveedores registrados en el factory:", ", ".join(ProviderFactory.list_available()))
provider = os.environ.get("LLM_DEFAULT", "anthropic")
role = os.environ.get("LLM_ROLE", "Eres un asistente breve que responde en español.")
print(f"Proveedor por defecto: {provider}")
print(f"Rol por defecto: {role}\n")

client = LLMClient(provider, system_prompt=role)

# 1) Listar modelos del proveedor activo (consulta en vivo /models)
try:
    print("Primeros modelos de", client.current.name, "->", client.list_models()[:5], "\n")
except LLMError as e:
    print(f"  No se pudieron listar modelos: {e}\n")

# 2) Una pregunta real al proveedor por defecto
print(f">>> Consultando a: {client.current.name}")
try:
    client.ask("Di 'hola' en una sola frase.")
except LLMError as e:
    print(f"  ERROR: {e}")
    print("  Revisa que en .env exista la API key del proveedor por defecto.")

# 3) Cambiar de proveedor en caliente (patrón Strategy) y repetir
other = [p for p in ProviderFactory.list_available() if p != client.current.name][0]
client.switch(other)
print(f"-> Cambiado a {other}\n")
print(f">>> Consultando a: {client.current.name}")
try:
    client.ask("Repite la palabra 'cambiado'.")
except LLMError as e:
    print(f"  ERROR: {e}  (sin key configurada para {other}?)")

## Para correr fuera del notebook (CLI)

```bash
python3 switcherllm.py
# o, si instalaste el paquete:
switcherllm
```

Archivos del proyecto:

```
lidr_1/
├── switcherllm.py            # Código cliente + REPL + punto de entrada
├── pyproject.toml            # Metadatos y dependencias del paquete
├── requirements.txt          # Dependencias para pip
├── .env.example              # Plantilla de credenciales (secretos, jamas comitearlos)
├── .gitignore                # .env, venv, __pycache__ fuera del repo
└── providers/
    ├── __init__.py           # Carga el .env automáticamente
    ├── base.py               # Interfaz común (Adapter target)
    ├── errors.py             # Errores normalizados + retry con backoff
    ├── pricing.py            # Coste estimado por llamada (tabla local de LLMPrice)
    ├── factory.py            # Crea el adaptador por nombre de proveedor
    ├── openai_provider.py    # Adaptador OpenAI
    ├── anthropic_provider.py # Adaptador Anthropic
    ├── gemini_provider.py    # Adaptador Google Gemini
    └── deepseek_provider.py  # Adaptador DeepSeek
```

Este notebook se genera automáticamente desde el código fuente (para no tener dos versiones que se desincronicen): `build_ipynb.py`.